In [6]:
import time
import os
import csv
import re
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from dotenv import load_dotenv

# ==========================================
# CONFIGURATION
# ==========================================
load_dotenv()
FACEBOOK_EMAIL = os.getenv('FACEBOOK_EMAIL')
FACEBOOK_PASSWORD = os.getenv('FACEBOOK_PASSWORD')

# Path Files
INPUT_LINKS_CSV = Path(r"CSV_file/Facebook_post_urls.csv")
OUTPUT_DETAILS_CSV = Path("scraped_full_content.csv")
PROFILE_PATH = Path(os.getcwd()) / 'fb_chrome_profile'

WAIT = 30

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def click_cookie(driver):
    """กดปิด Cookie Consent ถ้ามี"""
    sels = [
        'div[aria-label*="cookie"] span[dir="auto"]',
        'div[aria-label*="Cookie"] span[dir="auto"]',
        'button[data-cookiebanner="accept_button"]',
        'div[role="dialog"] button'
    ]
    for s in sels:
        try:
            btns = driver.find_elements(By.CSS_SELECTOR, s)
            for btn in btns:
                if "allow" in btn.text.lower() or "accept" in btn.text.lower() or "ยอมรับ" in btn.text:
                    btn.click()
                    return
        except: pass

def login(driver):
    driver.get("https://www.facebook.com/?locale=en_US")
    time.sleep(3)
    click_cookie(driver)
    
    if "login" in driver.current_url.lower():
        try:
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "email"))).send_keys(FACEBOOK_EMAIL)
            driver.find_element(By.ID, "pass").send_keys(FACEBOOK_PASSWORD)
            driver.find_element(By.NAME, "login").click()
            time.sleep(5)
        except:
            print("Login fields not found or already logged in.")

    # รอจนกว่าจะเข้าหน้าหลักได้
    try:
        WebDriverWait(driver, WAIT).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="main"]')))
    except:
        print("Warning: Main role not found after login, proceeding anyway...")

def expand_see_more(driver):
    """
    พยายามกด See more (ดูเพิ่มเติม) โดยใช้ JS
    """
    driver.execute_script("""
        var buttons = document.querySelectorAll('div[role="button"]');
        for(var i=0; i<buttons.length; i++){
            var t = buttons[i].innerText;
            if(t && (t.includes('See more') || t.includes('ดูเพิ่มเติม'))){
                buttons[i].click();
            }
        }
    """)
    time.sleep(1)

# ==========================================
# CORE EXTRACTION LOGIC
# ==========================================

def extract_content_precise(driver):
    """
    ดึงเนื้อหาโพสต์โดยเจาะจงเฉพาะส่วน Message Body
    และป้องกันการไปดึง Comment หรือ Related Post
    """
    script = """
        function getText() {
            // 1. หา Main Container
            var main = document.querySelector("div[role='main']");
            if (!main) return "";

            // 2. ลองหาจาก data-ad-preview='message' (แม่นยำสุด)
            var msgDiv = main.querySelector("div[data-ad-preview='message']");
            if (msgDiv) return msgDiv.innerText;

            // 3. ถ้าไม่เจอ ให้หา div ที่มี dir='auto' แต่อยู่ในส่วนบนของโพสต์
            // เทคนิค: ปกติเนื้อหาโพสต์จะอยู่ก่อนส่วนที่มีคำว่า 'Like'/'Comment'/'Share'
            
            // หาปุ่ม Action Bar (Like/Comment/Share) เพื่อใช้เป็นจุดตัด
            var actions = Array.from(main.querySelectorAll("div[role='button']")).find(el => 
                el.innerText.includes("Like") || el.innerText.includes("ถูกใจ") || 
                el.innerText.includes("Comment") || el.innerText.includes("แสดงความคิดเห็น")
            );

            var contentCandidate = "";
            var bestLength = 0;

            // กวาดหา div text ทั้งหมด
            var textDivs = main.querySelectorAll("div[dir='auto'], span[dir='auto']");
            
            for(var i=0; i<textDivs.length; i++) {
                var el = textDivs[i];
                
                // ถ้ามี Action Bar ให้เช็คว่า Text นี้อยู่ *เหนือ* Action Bar หรือไม่
                if (actions && (el.compareDocumentPosition(actions) & Node.DOCUMENT_POSITION_PRECEDING) === 0) {
                    continue; // ถ้าอยู่ใต้ปุ่ม Like แสดงว่าเป็น Comment -> ข้าม
                }

                var txt = el.innerText.trim();
                
                // กรอง Text ขยะ
                if(txt.length > 0 && 
                   !txt.includes("Like") && 
                   !txt.includes("Comment") && 
                   !txt.includes("Share") &&
                   !txt.match(/^\d+ (Comments|Shares)$/) &&
                   !txt.match(/^(All|Most relevant)$/) 
                ) {
                    // เลือก Text ที่ยาวที่สุดที่อยู่เหนือปุ่ม Like
                    if (txt.length > bestLength) {
                        bestLength = txt.length;
                        contentCandidate = txt;
                    }
                }
            }
            
            return contentCandidate;
        }
        return getText();
    """
    return driver.execute_script(script)

def extract_date_with_hover(driver):
    """ดึงวันที่โดยการ Hover (ใช้ Logic เดิมที่แก้ไปรอบที่แล้ว)"""
    try:
        script_find = """
            var all = document.querySelectorAll('a[role="link"]');
            for(var i=0; i<all.length; i++){
                var h = all[i].getAttribute('href');
                var l = all[i].getAttribute('aria-label');
                if(h && (h.includes('/posts/') || h.includes('/permalink/') || h.includes('multi_permalinks'))){
                    if(l) return all[i];
                }
            }
            return null;
        """
        el = driver.execute_script(script_find)
        
        if el:
            # Scroll ไปหาและ Hover
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
            time.sleep(0.5)
            ActionChains(driver).move_to_element(el).perform()
            time.sleep(0.5)
            
            val = el.get_attribute("aria-label")
            if not val: val = el.text
            return val
    except: pass
    return "N/A"

def get_post_data(driver, url):
    # Retry logic
    for attempt in range(2):
        try:
            driver.get(url)
            
            # รอโหลด Main
            WebDriverWait(driver, WAIT).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='main']")))
            time.sleep(3) # รอให้ Dynamic content นิ่ง
            
            # 1. กด See more
            expand_see_more(driver)
            
            # 2. ดึง Content (ด้วย Logic ใหม่)
            content = extract_content_precise(driver)
            
            # 3. ดึง Date
            date_str = extract_date_with_hover(driver)

            # Clean Text
            if content:
                content = content.replace("ดูน้อยลง", "").replace("See less", "").strip()

            # ถ้าได้ข้อมูลครบ หรือพยายามรอบ 2 แล้ว ให้ return เลย
            if (content and len(content) > 10) or attempt == 1:
                return content, date_str
            
        except Exception as e:
            print(f"Error on {url}: {e}")
            time.sleep(2)

    return "N/A", "N/A"

# ==========================================
# MAIN RUN
# ==========================================

if not FACEBOOK_EMAIL or not FACEBOOK_PASSWORD:
    raise SystemExit("Missing Env Vars")

# Setup Chrome
opts = uc.ChromeOptions()
opts.add_argument(f'--user-data-dir={PROFILE_PATH.as_posix()}')
opts.add_argument('--disable-notifications')
opts.add_argument('--lang=en-US')
opts.page_load_strategy = "eager"

# Version 144 ตามที่คุณแจ้งล่าสุด
driver = uc.Chrome(options=opts, version_main=144)
driver.set_page_load_timeout(60)

try:
    login(driver)
    
    if not INPUT_LINKS_CSV.exists():
        print("CSV file not found.")
        exit()

    with open(INPUT_LINKS_CSV, 'r', encoding='utf-8-sig') as infile, \
         open(OUTPUT_DETAILS_CSV, 'w', newline='', encoding='utf-8-sig') as outfile:
        
        reader = csv.DictReader(infile)
        writer = csv.writer(outfile)
        writer.writerow(['Post_URL', 'Full_Post_Content', 'Date'])
        
        # Limit 5 for testing
        count = 0
        for row in reader:
            if count >= 5: break
            
            url = (row.get('PostURL') or '').strip()
            if not url: continue
            
            print(f"[{count+1}] Scraping: {url}")
            
            txt, dt = get_post_data(driver, url)
            
            # Handle N/A
            if not txt: txt = "N/A"
            if not dt: dt = "N/A"
            
            writer.writerow([url, txt, dt])
            print(f"    -> Text Len: {len(txt)} | Date: {dt}")
            print("-" * 50)
            
            count += 1
            time.sleep(2) # Cool down

finally:
    driver.quit()

<>:92: SyntaxWarning: invalid escape sequence '\d'
<>:92: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_3138437/1022613477.py:92: SyntaxWarning: invalid escape sequence '\d'
  script = """


[1] Scraping: https://www.facebook.com/groups/homerentcm/posts/25388824220754690/
    -> Text Len: 132 | Date: N/A
--------------------------------------------------
[2] Scraping: https://www.facebook.com/groups/homerentcm/posts/25387194854250960/
    -> Text Len: 15 | Date: 3 วัน
--------------------------------------------------
[3] Scraping: https://www.facebook.com/groups/homerentcm/posts/25387220054248440/
    -> Text Len: 15 | Date: 3 วัน
--------------------------------------------------
[4] Scraping: https://www.facebook.com/groups/homerentcm/posts/25385663021070810/
    -> Text Len: 97 | Date: 35 นาที
--------------------------------------------------
[5] Scraping: https://www.facebook.com/groups/homerentcm/posts/25387839737519805/
    -> Text Len: 132 | Date: N/A
--------------------------------------------------
